In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import scipy

import plotly
from plotly.graph_objects import Scatter
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=False)
import plotly.express as px

import cosmosdr
from cosmosdr.plotting import basic_plot
import cosmosdr.signal_acquisition as s_acq
import cosmosdr.signal_processing as s_proc
import cosmosdr.plane_sailing as planes

from cosmosdr.plotting import create_base_figure

import structlog
logger = structlog.get_logger()

try:
    sdr.close()
except:
    pass

In [ ]:
ADSB_FREQUENCY = 1090e6
# Total number of bits in a single ADSB message
ADSB_BITS = 112
# One microsecond timeslot for one bit, 'on' if signal is within the first half of this slot
ADSB_SLOT_LENGTH = 1 / 1e6

center_freq = ADSB_FREQUENCY
# reccomended upper limit of sample rate. Fast enough to oversample
sample_rate = 2.4e6
n_reads = 32
n_samples = 4096

# choose integer samples per microsecond: 12 samples/us
# This is a good choice, because 6x2 = 12, and 5*2.4=12, so we can sample to 12, then subsample back to 1 block per 0.5us
target_sr = 12e6
samples_per_us_after_upsampling = int(round(target_sr/1e6))  # 12


### Sample at target SR, and upsample to higher rate

In [ ]:
# # Start up the SDR connection
# try:
#     sdr.close()
# except:
#     pass

# sdr = s_acq.get_sdr(center_freq=center_freq, sample_rate=sample_rate)

# s = s_acq.acquire_signal(sdr, n_reads=n_reads, n_samples=n_samples)

# highest_peak_read = s_acq.get_index_of_highest_peak(s)

In [ ]:

iq = planes.get_example_dataset(center_freq=center_freq, sample_rate=sample_rate, n_reads=n_reads, n_samples=n_samples)

In [ ]:
basic_plot(iq)

In [ ]:
iq.shape

In [ ]:
iq_resampled, _ = s_proc.resample_to_target(iq, sample_rate, target_sr)
iq_mag = s_proc.iq_to_envelope(iq_resampled)

In [ ]:
iq_mag.shape

In [ ]:
basic_plot(iq_mag)

In [ ]:
iq_mag = planes.trim_iq_around_peak(iq_mag)

In [ ]:
basic_plot(iq_mag)

In [ ]:
iq_mag = planes.shift_to_optimal_phase(iq_mag, samples_per_us=samples_per_us_after_upsampling)

In [ ]:
iq_mag = planes.downsample_to_buckets(iq_mag, samples_per_us_after_upsampling)

In [ ]:
basic_plot(iq_mag)

# Interpreting aircraft signals

In [ ]:
from copy import copy

In [ ]:
# TODO infer this from the data
threshold = 0.21

In [ ]:
basic_plot(np.sort(iq_mag))

In [ ]:
s_bin = (s > threshold).astype(int)

In [ ]:
# mode S preamble, 8us
preamble = [1,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0]
len_preamble = len(preamble)
assert(len_preamble==16)
    

In [ ]:
basic_plot(s_bin)

In [ ]:
for i in range(len(s_bin)):
    candidate = s_bin[i:i+len_preamble]
    if np.array_equal(candidate, preamble):
        logger.info("ADSB preamble identified @ idx: %s, damn!", i)
        break

In [ ]:
i

In [ ]:
# the index of the last piece of the preamble
i+len_preamble

In [ ]:
# Drop the preamble off the front of the burst
adsb = s_bin[i+len_preamble:]

In [ ]:
basic_plot(adsb)

In [ ]:
# Note, the last part of the signal may indeed be one or many zeros, if this doens't fit into a standard msg length, that's likely why
adsb = np.trim_zeros(adsb)

In [ ]:
# If we end up with exactly one less bit than was expected, then assume we just chopped off a bit
if (len(adsb) + 1) == ADSB_BITS * 2:
    adsb = np.append(adsb, 0)

In [ ]:
adsb

In [ ]:
# Take every other value (a.k.a. take the value in teh first half of each bucket
# This is the same as 'if there is a pule in the first half, then 1.'
adsb = adsb[::2]

### Assess whether it's ELS (extended length message) or not

In [ ]:
len(adsb[5:8])

In [ ]:
ADSB_DF = 0, 5
ADSB_CAPABILITY = 5, 8  # transponder level, ranking 1 - 8 in binary
ADSB_ICAO_ADDR = 8, 32  # Unique identifier for the aircraft
ADSB_MSG = 32, 88
ADSB_PARITY_PI = 88, 112

MSG = [
    ADSB_DF,
    ADSB_CAPABILITY,
    ADSB_ICAO_ADDR,
    ADSB_MSG,
    ADSB_PARITY_PI,
]

In [ ]:
# Ensure the msg is broken up correctly
assert(sum([len(adsb[start:end]) for start, end in MSG]) == len(adsb))

In [ ]:
def parse_segment(adsb, segment: tuple[int, int]):
    return adsb[segment[0]:segment[1]]

def convert_binary_array_to_int(a):
    """Use builtin in casting, first convert the array to a string, e.g. '101010' """
    return int(''.join(map(str, a)), 2)
    

In [ ]:
adsb_msg = parse_segment(adsb, ADSB_MSG)

In [ ]:
adsb_msg_type_code = adsb_msg[:5]

In [ ]:
convert_binary_array_to_int(adsb_msg_type_code)

In [ ]:
"""
1–4 	Aircraft identification
5–8 	Surface position
9–18 	Airborne position (w/Baro Altitude)
19 	Airborne velocities
20–22 	Airborne position (w/GNSS Height)
23–27 	Reserved
28 	Aircraft status
29 	Target state and status information
31 	Aircraft operation status
"""

In [ ]:
def identify_msg_type(adsb_msg):
    

# Plot the on/off signalling

- If the peak is one colour, then the signal was in the first half of a slot
- If the peak is the other,  then the signal was in the second half of a slot

On or off depends on the start point, can't be predicted assessed ahead of time

In [ ]:
evens_indexer = signal_col.index % 2 == 0

In [ ]:
# Split the first half/second half into separate columns so they can be plotted differently
even = signal_col.reindex(signal_col.index[evens_indexer])
odd  = signal_col.reindex(signal_col.index[~evens_indexer])
even.name="even"
odd.name="odd"

# Plot the strongest signal to highlight 1s and 0s
- Within each 1us (microsecond, millionth of a second), there are two halves to the 'frame'
- If there is a signal peak within the first half, this is a 1
- If there is a signal peak within the second half, this is a 0

In [ ]:
# fig = px.bar(pd.concat([even, odd], axis=1))

# fig.update_traces(marker_line_width = 0,
#                   selector=dict(type="bar"))

In [ ]:
low_cut = 0.15